# Observabilidad, Metadata y Linaje para Ecosistemas IA/Big Data (Simulación)

Objetivo: simular una tubería de datos con eventos de cambio (schema, frescura, errores), modelar linaje de datasets y jobs, y generar un **reporte de impacto** para gobernanza y auditoría.

Cubrimos: métricas (freshness, latency, error-rate), logs, alertas; gestión de metadatos mínimos; linaje table-level/column-level (simplificado).

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Mini Data Mesh: Dos Dominios con Catálogo Compartido](17_mini_data_mesh_dos_dominios_catalogo_compartido.ipynb) | Siguiente: [Caso Práctico: Streaming + Inferencia en Tiempo Real](19_streaming_inferencia_tiempo_real_caso_practico.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. 1. Modelo simple de linaje y metadatos
2. 2. Eventos operacionales y métricas (observabilidad)
3. 3. Evento de cambio de esquema y reporte de impacto
4. 4. Linaje y análisis de impacto por frescura
5. 5. Logs y auditoría mínima
6. 6. Reporte final (gobernanza)
7. Actualización 2025: Observabilidad, Metadata y Linaje Moderno

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Observabilidad, Metadata y Linaje para Ecosistemas IA/Big Data (Simulación)"] --> C["Notebook siguiente"]
```


## 1. Modelo simple de linaje y metadatos
Representamos datasets, jobs y dependencias. Incluimos columnas relevantes por dataset para un trazo de impacto por cambios de esquema.

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Set
import pandas as pd

@dataclass
class Dataset:
    name: str
    columns: Set[str] = field(default_factory=set)
    domain: str = ''
    owner: str = ''

@dataclass
class Job:
    name: str
    inputs: List[str]
    outputs: List[str]
    code_ref: str = ''  # notebook/file

class Lineage:
    def __init__(self):
        self.datasets: Dict[str, Dataset] = {}
        self.jobs: Dict[str, Job] = {}
        self.parents: Dict[str, Set[str]] = {}   # dataset -> parents datasets
        self.children: Dict[str, Set[str]] = {}  # dataset -> children datasets
    def add_dataset(self, ds: Dataset):
        self.datasets[ds.name] = ds
        self.parents.setdefault(ds.name, set())
        self.children.setdefault(ds.name, set())
    def add_job(self, job: Job):
        self.jobs[job.name] = job
        for o in job.outputs:
            self.parents.setdefault(o, set()).update(job.inputs)
        for i in job.inputs:
            self.children.setdefault(i, set()).update(job.outputs)
    def impact_upstream(self, dataset: str) -> Set[str]:
        # retorna todos los downstream afectados por cambios en dataset
        impacted = set()
        stack = [dataset]
        while stack:
            cur = stack.pop()
            for ch in self.children.get(cur, []):
                if ch not in impacted:
                    impacted.add(ch)
                    stack.append(ch)
        return impacted
    def find_dependents_by_column(self, source_ds: str, column: str) -> List[str]:
        # detecta datasets descendientes que declaran depender de la columna
        impacted = []
        for ds_name in self.impact_upstream(source_ds):
            ds = self.datasets.get(ds_name)
            if ds and column in ds.columns:
                impacted.append(ds_name)
        return impacted

# Definimos un pipeline típico IA/Big Data
L = Lineage()

# Bronze
L.add_dataset(Dataset('bronze.sales_orders_raw', {'order_id','order_amount','order_date','customer_id'},'sales','owner_sales'))
L.add_dataset(Dataset('bronze.marketing_campaigns_raw', {'campaign_id','channel','impressions','clicks','spend'},'marketing','owner_marketing'))

# Jobs a Silver
L.add_job(Job('clean_sales_orders',[ 'bronze.sales_orders_raw' ],[ 'silver.orders_clean' ], code_ref='02-nivel-mid/18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb'))
L.add_job(Job('clean_marketing_campaigns',[ 'bronze.marketing_campaigns_raw' ],[ 'silver.campaigns_clean' ], code_ref='02-nivel-mid/18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb'))

# Silver datasets
L.add_dataset(Dataset('silver.orders_clean', {'order_id','order_amount','order_date','customer_id','channel'}))
L.add_dataset(Dataset('silver.campaigns_clean', {'campaign_id','channel','impressions','clicks','spend'}))

# Gold y features
L.add_job(Job('join_sales_campaigns',[ 'silver.orders_clean','silver.campaigns_clean' ],[ 'gold.bi_sales_campaign_join' ], code_ref='02-nivel-mid/18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb'))
L.add_dataset(Dataset('gold.bi_sales_campaign_join', {'order_id','order_amount','order_date','customer_id','channel','campaign_id','impressions','clicks','spend'}))

L.add_job(Job('build_orders_features',[ 'silver.orders_clean' ],[ 'features.orders_features' ], code_ref='02-nivel-mid/18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb'))
L.add_dataset(Dataset('features.orders_features', {'customer_id','orders_30d','revenue_30d'}))

# Model consumer
L.add_job(Job('train_model_reco',[ 'features.orders_features' ],[ 'models.reco_v1' ], code_ref='02-nivel-mid/18_observabilidad_metadata_linaje_ia_bigdata_simulacion.ipynb'))
L.add_dataset(Dataset('models.reco_v1', {'model_artifact','metrics_auc'}))

print('Datasets:', len(L.datasets), '| Jobs:', len(L.jobs))

In [ ]:
# Visualización de grafo de linaje (opcional)
# Requiere networkx y matplotlib (añadir 'networkx' al requirements.txt si no está presente)
try:
    import networkx as nx
    import matplotlib.pyplot as plt
    G = nx.DiGraph()
    for ds, obj in L.datasets.items():
        G.add_node(ds, type='dataset')
    for job in L.jobs.values():
        for i in job.inputs:
            for o in job.outputs:
                G.add_edge(i, o, job=job.name)
    plt.figure(figsize=(10,6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw_networkx_nodes(G, pos, node_size=700, node_color='#4C78A8')
    nx.draw_networkx_labels(G, pos, font_size=8)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowstyle='->', arrowsize=12)
    plt.title('Lineage (datasets y flujo)')
    plt.axis('off')
    plt.show()
except ImportError:
    print('networkx/matplotlib no instalados: omitiendo visualización.')

## 2. Eventos operacionales y métricas (observabilidad)
Simulamos freshness (minutos), error-rate, throughput y generamos alertas por umbral.

In [ ]:
from datetime import datetime, timedelta
import random
random.seed(7)

metrics = {
    'bronze.sales_orders_raw': {'freshness_m': 20, 'error_rate': 0.002, 'throughput_rps': 120},
    'bronze.marketing_campaigns_raw': {'freshness_m': 75, 'error_rate': 0.0, 'throughput_rps': 90},
    'silver.orders_clean': {'freshness_m': 25, 'error_rate': 0.005, 'throughput_rps': 300},
    'silver.campaigns_clean': {'freshness_m': 80, 'error_rate': 0.0, 'throughput_rps': 220},
    'gold.bi_sales_campaign_join': {'freshness_m': 85, 'error_rate': 0.01, 'throughput_rps': 150},
}

thresholds = { 'freshness_m_max': 60, 'error_rate_max': 0.01 }

alerts = []
for ds, m in metrics.items():
    if m['freshness_m'] > thresholds['freshness_m_max']:
        alerts.append((ds, 'freshness', m['freshness_m']))
    if m['error_rate'] > thresholds['error_rate_max']:
        alerts.append((ds, 'error_rate', m['error_rate']))

print('Alertas generadas:', alerts)

## 3. Evento de cambio de esquema y reporte de impacto
Caso: renombramos `order_amount` → `amount` en `bronze.sales_orders_raw`. Detectamos descendientes afectados (column-level simplificado).

In [ ]:
# Cambio de esquema en la fuente
source = 'bronze.sales_orders_raw'
old_col, new_col = 'order_amount', 'amount'
print('Schema change:', source, old_col, '→', new_col)

# Aplicamos el cambio en el dataset fuente
if old_col in L.datasets[source].columns:
    L.datasets[source].columns.remove(old_col)
    L.datasets[source].columns.add(new_col)

# Descendientes que dependen explícitamente de la columna antigua
dependientes = L.find_dependents_by_column(source, old_col)
report = pd.DataFrame({'impacted_dataset': dependientes})
report

### 3.1 Recomendaciones automáticas
- Actualizar transformaciones en `clean_sales_orders` y aguas abajo.
- Agregar regla de compatibilidad (alias `order_amount`→`amount`) temporal.
- Notificar owners de `silver.orders_clean`, `gold.bi_sales_campaign_join`, `features.orders_features`.

## 4. Linaje y análisis de impacto por frescura
Cuando `bronze.marketing_campaigns_raw` supera frescura de 60m, los consumidores `silver.campaigns_clean` y `gold.bi_sales_campaign_join` heredan el retraso.

In [ ]:
stale_source = 'bronze.marketing_campaigns_raw'
down = L.impact_upstream(stale_source)
impact_fresh = pd.DataFrame({
    'downstream': list(sorted(down)),
    'note': 'posible retraso por fuente con freshness>60m'
})
impact_fresh

## 5. Logs y auditoría mínima
Registramos eventos con timestamp, tipo, severidad y entidad afectada (dataset/job).

In [ ]:
from datetime import datetime
logs = []
def log(event_type, severity, entity, details):
    logs.append({'ts': datetime.utcnow().isoformat(), 'type': event_type, 'sev': severity, 'entity': entity, 'details': details})

log('schema_change','high', source, f'{old_col}-> {new_col}')
for ds, kind, val in alerts:
    log('alert', 'medium', ds, f'{kind}={val}')
pd.DataFrame(logs)

## 6. Reporte final (gobernanza)
Incluye: alertas activas, datasets impactados por esquema y por frescura, y próximos pasos.

In [ ]:
reporte = {
  'alerts': alerts,
  'schema_change_source': source,
  'schema_impacted': dependientes,
  'freshness_source': stale_source,
  'freshness_impacted': list(sorted(down)),
  'recommendations': [
      'Agregar alias de columna y deprecación programada',
      'Actualizar contratos de datos y avisar a consumidores',
      'Asegurar pruebas de compatibilidad y dashboards de frescura'
  ]
}
pd.Series(reporte)

## Actualización 2025: Observabilidad, Metadata y Linaje Moderno

### Panorama 2025
La observabilidad de datos evoluciona de métricas aisladas a una capa integrada que combina calidad, linaje en tiempo (casi) real, monitoreo de ML, gobernanza y FinOps. El objetivo: reducir MTTR ante incidentes y optimizar costo-riesgo.

### Ecosistema de Herramientas (selección validada)
- Recolección / Instrumentación: OpenTelemetry (trazas, métricas), Spark Listener, dbt artifacts (manifest.json, run_results.json), Airflow lineage backend, Delta Lake Change Data Feed, Iceberg snapshots, Hudi timeline.
- Linaje: OpenLineage (estándar de eventos), Marquez (backend), DataHub (lineage + metadata + búsquedas), Apache Atlas, Amundsen (legacy), Atlas/Purview/Glue/GCP Data Catalog (catálogo + clasificación).
- Calidad & DQ: Great Expectations, Soda, dbt tests, Monte Carlo (observabilidad gestionada), Bigeye.
- Observabilidad GenAI/ML: Evidently AI (drift, performance), WhyLabs (anomaly streams), MLflow (experimentos + modelos), Feast (features + monitoreo), FeatureByte / Tecton (feature lifecycle).
- Métricas & Dashboards: Prometheus, Grafana, SigNoz (APM OSS), OpenMetric exporters.

### Categorías de Métricas Clave
| Dimensión | Métricas Ejemplo | Objetivo 2025 |
|-----------|------------------|---------------|
| Freshness / Actualización | latency_ingest_s, data_freshness_minutes | < 15 min (zonas críticas) |
| Completitud | completeness_pct, expected_rows_vs_actual | > 99% |
| Calidad | null_rate, duplicate_rate, distribution_drift_ks_pvalue | null < 2%, duplicados < 1% |
| Esquema | schema_drift_events, breaking_change_count | 0 breaking por trimestre |
| Linaje Cobertura | lineage_coverage_pct (assets con linaje) | > 85% |
| Propiedad / Ownership | assets_with_owner_pct | > 95% |
| ML Drift | prediction_drift_psi, feature_drift_topN | PSI < 0.2 |
| FinOps | cost_pipeline_daily_usd, waste_pct, idle_cluster_hours | waste < 10% |
| SLA / Fiabilidad | sla_met_pct, failed_runs, mttr_minutes | SLA > 99%, MTTR < 30 |
| Seguridad / PII | pii_fields_classified_pct, masked_access_events | > 98% clasificados |

### Flujo de Linaje Estándar (OpenLineage)
1. Job inicia (programador / orquestador emite "START").
2. Se capturan datasets de entrada/salida (nombres, namespace, facetas de esquema).
3. Se publican eventos a backend (Marquez o DataHub) vía API.
4. Dashboards calculan cobertura y alertas (ej. schema_drift_events > 0).

### Ejemplo de Emisión de Evento OpenLineage (Python)
```python
from openlineage.client import OpenLineageClient
from openlineage.client.run import RunEvent, RunState, Run, Job, Dataset
from datetime import datetime

client = OpenLineageClient(url="http://marquez:5000")
run_id = "mid-obs-ejemplo-2025"

input_ds = Dataset(namespace="postgres.public", name="sales_orders")
output_ds = Dataset(namespace="delta.lakehouse", name="ventas_limpias")

run_event = RunEvent(
    eventType=RunState.START,
    eventTime=datetime.utcnow().isoformat()+"Z",
    run=Run(runId=run_id),
    job=Job(namespace="curso.mid", name="transform_ventas"),
    inputs=[input_ds],
    outputs=[output_ds]
)
client.emit(run_event)
```
(Se repetirían eventos para `COMPLETE` y, si ocurre, `FAIL`.)

### Data Quality / Drift Rápido (Distribución & PSI)
```python
import pandas as pd, numpy as np
from scipy.stats import ks_2samp

ref = pd.read_parquet("/data/zona_silver/ventas_ref.parquet")
cur = pd.read_parquet("/data/zona_silver/ventas_hoy.parquet")

feature = "total_amount"
ks_p = ks_2samp(ref[feature], cur[feature]).pvalue
psi_bins = np.linspace(min(ref[feature].min(), cur[feature].min()),
                       max(ref[feature].max(), cur[feature].max()), 11)
ref_dist, _ = np.histogram(ref[feature], bins=psi_bins)
cur_dist, _ = np.histogram(cur[feature], bins=psi_bins)
ref_pct = ref_dist / ref_dist.sum(); cur_pct = cur_dist / cur_dist.sum()
psi = np.sum((cur_pct - ref_pct) * np.log((cur_pct+1e-9)/(ref_pct+1e-9)))
print({"ks_pvalue": ks_p, "psi": psi})
```

### Integración FinOps en Observabilidad
- Asociar cada pipeline a etiquetas: `team`, `env`, `sla_tier`.
- Exponer costo por ejecución + costo acumulado mensual.
- Métrica waste: horas de cluster activo sin jobs / horas totales.
- Alertas: waste_pct > umbral, costo_diario > forecast.

### Mapa de Facetas Metadata (Ejemplos)
| Faceta | Ejemplo | Uso |
|--------|---------|-----|
| Schema Facet | columnas, tipos | Validar drift y compatibilidad |
| Data Quality Facet | reglas evaluadas, resultados | Alertas tempranas |
| Statistics Facet | count, min, max, avg | Observación de anomalías |
| Ownership Facet | owner, domain, stewardship | Gobernanza y accountability |
| Classification Facet | etiquetas PII, confidencial | Seguridad / Cumplimiento |
| Cost Facet | costo_ejecucion, costo_storage | FinOps optimización |

### Ejercicio 2025 (Aplicar Sobre Dataset Sintético)
1. Calcular `null_rate` y `duplicate_rate` sobre `sales_orders.csv`.
2. Emular evento de linaje para transformación hacia vista enriquecida.
3. Generar comparación de distribución (KS + PSI) para `total_amount` vs histórico.
4. Definir 3 alertas accionables (ej. PSI > 0.25, freshness_minutes > 30, waste_pct > 15%).
5. Registrar resultados en diccionario normalizado para posterior envío (simulado) a un endpoint de observabilidad.

### Checklist de Implementación 2025
- [ ] Linaje: eventos START/COMPLETE instrumentados para >80% jobs críticos.
- [ ] Calidad: >=10 reglas automatizadas (null, duplicados, rangos, drift). 
- [ ] ML/Feature Drift: monitoreo PSI/KS en features clave y predicciones.
- [ ] Costos: dashboard con costo por pipeline y waste_pct diario.
- [ ] Gobernanza: >90% assets con owner + clasificación PII aplicada.
- [ ] Alertas semánticas: thresholds documentados en ADR de observabilidad.

### Maturity Model Observabilidad (Niveles)
| Nivel | Rasgos | Prioridad Mejora |
|-------|--------|------------------|
| 1 (Básico) | Métricas manuales, sin linaje | Instrumentar eventos OpenLineage |
| 2 (Inicial) | Calidad parcial, dashboard estático | Automatizar reglas DQ y freshness |
| 3 (Intermedio) | Linaje principal + tests dbt | Añadir drift ML y FinOps básicos |
| 4 (Avanzado) | Cobertura >70%, alertas proactivas | Expandir clasificación + ownership |
| 5 (Optimizado) | Ajuste costo-riesgo continuo | Experimentar predicción de incidentes |

### Referencias Verificadas
- OpenLineage (https://openlineage.io)
- DataHub (https://datahubproject.io)
- Delta Lake CDF (https://docs.delta.io/latest/delta-change-data-feed.html)
- Evidently AI (https://www.evidentlyai.com)
- Prometheus (https://prometheus.io) / Grafana (https://grafana.com)
- Apache Atlas (https://atlas.apache.org)
- MLflow (https://mlflow.org)
- Feast (https://feast.dev)
- Soda (https://soda.io)

> Mantener referencias vivas: revisar trimestralmente adopción y deprecar herramientas obsoletas.


### Referencias Anexos (Recursos Reutilizables 2025)

Artefactos para extender ejercicios de observabilidad:
- Contrato Datos: `04-recursos/anexos/data_contract_template.yaml`
- ADR Template: `04-recursos/anexos/adr_template.md`
- Política Column Access (Rego): `04-recursos/anexos/column_access.rego`
- Emisor Linaje OpenLineage: `04-recursos/helpers/event_emitter.py`
- Métricas Calidad / Drift: `04-recursos/helpers/quality_rules.py`
- Costos / Waste / Ratio: `04-recursos/helpers/cost_metrics.py`
- Catálogo KPIs: `04-recursos/kpi_catalog.yml`

> Integrar estos recursos en pipelines reales sin duplicar definiciones.


<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- OpenLineage: https://openlineage.io/
- DataHub: https://datahubproject.io/
- NIST AI Risk Management Framework: https://www.nist.gov/itl/ai-risk-management-framework
- Referencias del curso: ../04-recursos/referencias.md
